<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

Figure 1.1.a. Jupyter environment + Python notebooks

# Digital Earth Pacific Notebook 1B prepare postcard and load data to csv

The objective of this notebook is to prepare a geomad postcard for your AOI (masking, scaling and loading additional band ratios and spectral indices) and sampling all the datasets into a csv based on your training data geodataframe.

## Step 1.1: Configure the environment

In [35]:
import os
from datetime import datetime
from shapely.geometry import Polygon
from shapely import box
from pyproj import CRS 
import folium
from odc.geo import Geometry
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio as rio
import xarray as xr
import rioxarray
from ipyleaflet import basemaps
from numpy.lib.stride_tricks import sliding_window_view
import pystac_client
from dask.distributed import Client as DaskClient
from odc.stac import load, configure_s3_access
import planetary_computer
# from odc.stac import load
from pystac.client import Client
from skimage.feature import graycomatrix, graycoprops
from utils import load_data, load_s1_dem, scale, calculate_band_indices, apply_mask, mask_land, all_masks, do_prediction

In [19]:
# Reload scripts and imports
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<font color='red'>1.1. Revise the site of training data...


In [20]:
site = "nasese"
filename = f"training-data/{site}-processed-tdata.geojson"

## Step 1.2: Configure STAC access and search parameters

In [21]:
catalog = "https://stac.digitalearthpacific.org"
client = Client.open(catalog)

<font color='red'>1.2. Revise the site

In [22]:
area_of_interest = "nasese"

# 1. Read the file
aoi = gpd.read_file(f"{filename}")

# 2. Get the bounding box directly from the GeoDataFrame
# This returns exactly what STAC needs: (minx, miny, maxx, maxy)
bbox = aoi.total_bounds 

<font color='red'>1.3. Revise the year to '2025'

In [23]:
# 3. Proceed with your search
year = '2025'
items = list(
    client.search(
        collections=["dep_s2_geomad"],
        bbox=bbox,
        datetime=year
    ).items()
)

#### How many images were found?

In [24]:
print(f"Found {len(items)} items in for {datetime}")

Found 1 items in for <class 'datetime.datetime'>


<font color='red'>1.4. Define the spectral bands you are interested in as:
> `measurements = ["nir", "red", "blue", "green", "emad", "smad", "bcmad", "green", "nir08", "nir09", "swir16", "swir22", "coastal", "rededge1", "rededge2", "rededge3"]`

In [25]:
measurements = ["nir", "red", "blue", "green", "emad", "smad", "bcmad", "green", "nir08", "nir09", "swir16", "swir22", "coastal", "rededge1", "rededge2", "rededge3"]

<font color='red'>1.5. Load the data

In [26]:
data = load_data(
    items,
    measurements,
    bbox,
)

In [27]:
# dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit='32GB')
# configure_s3_access(cloud_defaults=True, requester_pays=True)

<font color='red'>1.6. Scale the variable 'data'
>scaled = scale(`insertvariable`)

In [28]:
scaled = scale(data)

In [29]:
scaled = scaled.squeeze()

<font color='red'>1.7. Explore the new 'scaled' variable 
>`insertvariable`.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

In [3]:
scaled.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

<font color='red'>1.8. Calculate the spectral bands for this 'scaled' variable 
>`variable` = calculate_band_indices(`variable`)  

In [31]:
scaled = calculate_band_indices(scaled)

In [32]:
# see results by typing inserting variable and running code below
scaled

<xarray.Dataset> Size: 13MB
Dimensions:        (y: 215, x: 251)
Coordinates:
  * y              (y) float64 2kB -2.042e+06 -2.042e+06 ... -2.044e+06
  * x              (x) float64 2kB 3.165e+06 3.165e+06 ... 3.168e+06 3.168e+06
    spatial_ref    int32 4B 3832
    time           datetime64[ns] 8B 2025-01-01
Data variables: (12/32)
    nir            (y, x) float64 432kB dask.array<chunksize=(215, 251), meta=np.ndarray>
    red            (y, x) float64 432kB dask.array<chunksize=(215, 251), meta=np.ndarray>
    blue           (y, x) float64 432kB dask.array<chunksize=(215, 251), meta=np.ndarray>
    green          (y, x) float64 432kB dask.array<chunksize=(215, 251), meta=np.ndarray>
    emad           (y, x) float32 216kB dask.array<chunksize=(215, 251), meta=np.ndarray>
    smad           (y, x) float32 216kB dask.array<chunksize=(215, 251), meta=np.ndarray>
    ...             ...
    ndci           (y, x) float64 432kB dask.array<chunksize=(215, 251), meta=np.ndarray>
    nbi            (y, x) float64 432kB dask.array<chunksize=(215, 251), meta=np.ndarray>
    ndmi           (y, x) float64 432kB dask.array<chunksize=(215, 251), meta=np.ndarray>
    bsi            (y, x) float64 432kB dask.array<chunksize=(215, 251), meta=np.ndarray>
    awei           (y, x) float64 432kB dask.array<chunksize=(215, 251), meta=np.ndarray>
    tc_wetness     (y, x) float64 432kB dask.array<chunksize=(215, 251), meta=np.ndarray>

<font color='red'>1.9. Mask out the 'scaled' variable including the ocean 
>`scaled`, mask = all_masks(`scaled`, return_mask = True)


In [36]:
scaled, mask = all_masks(scaled, return_mask = True)

<font color='red'>Explore the extent of this 'mask' variable  
>`insertvariable`.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

In [2]:
scaled.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

### Postcard csv

The objective of this notebook was to train the machine learning model that will allow us to classify an area with land cover classes defined through the training data.

Step 1.2. Input the training data to sample geomad data from the postcard.

`training = gpd.read_file(f"training-data/{site}-processed-tdata.geojson")`

In [38]:
training = gpd.read_file(f"training-data/{site}-processed-tdata.geojson")

<font color='red'>2.1. Explore the new clipped extent of the 'training' data within the extent of your aoi  
> `variable`.explore()

In [1]:
training.explore()

In [40]:
# Reproject training data to the GeoMAD CRS and convert to xarray
training_reprojected = training.to_crs(scaled.odc.crs)
training_da = training_reprojected.assign(
    x=training_reprojected.geometry.x, y=training_reprojected.geometry.y
).to_xarray()

# Extract training values from the masked dataset
training_values = (
    scaled.sel(training_da[["x", "y"]], method="nearest")
    .squeeze()
    .compute()
    .to_pandas()
)
# training_values

<font color='red'>2.2. You have made a new variable called 'training values'... To better understand what is inside this inspect the columns  
> `variable`.columns

In [42]:
training.columns

Index(['Category', 'LULC', 'Coastal', 'Surveyor', 'Date', 'Notes', 'LULC_code',
       'Marine_code', 'geometry'],
      dtype='object')

<font color='red'>2.3. Run the following lines of code unchanged to generate more information about the data array you are using to train the machine learning model - insert "LULC_code" inside the training[...] as this is what the ML model will learn from. Note you will need to change the name of the column to look at marine habitats

In [47]:
# Join the training data with the extracted values and remove unnecessary columns
training_array = pd.concat([training["Marine_code"], training_values], axis=1)

# Drop rows where there was no data available
training_array = training_array.dropna()

# Preview our resulting training array
# training_array.head()

In [48]:
print(training_array.shape[1], 'total columns')
# print('columns included', training_array.columns)

37 total columns


In [50]:
print(training_array['Marine_code'].value_counts())
print('total gps points',(len(training_array)))

Marine_code
2.0    452
4.0    290
1.0    139
8.0     39
7.0     34
3.0     15
5.0     13
6.0      6
Name: count, dtype: int64
total gps points 988


In [51]:
training_array=training_array.drop(columns=["spatial_ref", "time"])

<font color='red'>2.4. You have made a final `training_array` variable. Inspect it here by entering it below:

In [4]:
training_array

<font color='red'>2.5. Export your final `training_array` variable to a CSV file.
>`variable`.to_csv(f"training-data/{site}-tdata.csv", index=False)

In [53]:
training_array.to_csv(f"training-data/{site}-tdata.csv", index=False)